In [0]:
%python
### dml with dlta table

In [0]:
%sql
create table delta_catalog.raw.ext_table_dml(
  id int,
  order_name string,
  amount double,
  prod_id int
)
using delta
location 'abfss://raw@storagedeltalakepoorna.dfs.core.windows.net/ext_table_dml'

In [0]:
%sql
insert into delta_catalog.raw.ext_table_dml
values (1, 'order1', 1001, 100),
       (2, 'order2', 2002, 200),
       (3, 'order3', 3003, 300)

num_affected_rows,num_inserted_rows
3,3


In [0]:
%python
### turning off deletion vectors


In [0]:
%sql
-- For Delta table
ALTER TABLE delta_catalog.raw.ext_table_dml SET TBLPROPERTIES ('delta.enableDeletionVectors' = false);

-- For Iceberg tables, use iceberg.enableDeletionVectors instead of delta.enableDeletionVectors

In [0]:
%python
###update in delta table 

In [0]:
%sql
update delta_catalog.raw.ext_table_dml set amount = 1000 where id = 1

num_affected_rows
1


In [0]:
%sql
select * from delta_catalog.raw.ext_table_dml

id,order_name,amount,prod_id
1,order1,1000.0,100
2,order2,2002.0,200
3,order3,3003.0,300


In [0]:
%sql
delete from delta_catalog.raw.ext_table_dml where id = 1

num_affected_rows
1


In [0]:
%python
 ###time travel and versioning

In [0]:
%sql
select * from delta_catalog.raw.ext_table_dml

id,order_name,amount,prod_id
2,order2,2002.0,200
3,order3,3003.0,300


In [0]:
%sql
describe history delta_catalog.raw.ext_table_dml
    

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-04-21T06:58:03.000Z,144173091364030,22pa1a42a9@vishnu.edu.in,DELETE,"Map(predicate -> [""(id#15970 = 1)""])",null,List(315579491889577),fb2f8703-2340-45cc-8126-eea0e13e48a9,0421-041435-e34bkrji-v2n,3,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1259, numCopiedRows -> 2, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1091, numDeletionVectorsUpdated -> 0, numDeletedRows -> 1, scanTimeMs -> 419, numAddedFiles -> 1, numAddedBytes -> 1240, rewriteTimeMs -> 663)",null,Databricks-Runtime/18.1.x-photon-scala2.13
3,2026-04-21T06:39:21.000Z,144173091364030,22pa1a42a9@vishnu.edu.in,UPDATE,"Map(predicate -> [""(id#15650 = 1)""])",null,List(315579491889577),cc2a03c0-92c2-409f-b8e7-f64a1392a160,0421-041435-e34bkrji-v2n,2,WriteSerializable,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 1235, numCopiedRows -> 2, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 1208, numDeletionVectorsUpdated -> 0, scanTimeMs -> 504, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1259, rewriteTimeMs -> 687)",null,Databricks-Runtime/18.1.x-photon-scala2.13
2,2026-04-21T06:37:56.000Z,144173091364030,22pa1a42a9@vishnu.edu.in,SET TBLPROPERTIES,"Map(properties -> {""delta.enableDeletionVectors"":""false""})",null,List(315579491889577),9602850f-1d64-4c49-aad0-921e6484052b,0421-041435-e34bkrji-v2n,1,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-photon-scala2.13
1,2026-04-21T06:34:57.000Z,144173091364030,22pa1a42a9@vishnu.edu.in,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(315579491889577),c4a7a2e3-8faa-4aca-a973-f97f946d7db1,0421-041435-e34bkrji-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 1235)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-04-21T06:34:35.000Z,144173091364030,22pa1a42a9@vishnu.edu.in,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(315579491889577),9d879b76-358e-4b3d-9b65-605f9faafd6b,0421-041435-e34bkrji-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/18.1.x-photon-scala2.13


In [0]:
%sql
restore delta_catalog.raw.ext_table_dml to version as of 4

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
1240,1,1,1,1259,1240


In [0]:
%sql
select * from delta_catalog.raw.ext_table_dml

id,order_name,amount,prod_id
2,order2,2002.0,200
3,order3,3003.0,300


In [0]:
%python
###optimizw

In [0]:
%sql
optimize delta_catalog.raw.ext_table_dml zorder by id

path,metrics
abfss://raw@storagedeltalakepoorna.dfs.core.windows.net/ext_table_dml,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 1388), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1776766873904, 1776766874290, 8, 0, null, List(0, 0), null, 4, 4, 0, 0, null, null)"


In [0]:
create table if not exists delta_catalog.raw.liq_table(
  id int,
  amount double,
  name string
)
using delta
location 'abfss://raw@storagedeltalakepoorna.dfs.core.windows.net/liq_table'
cluster by (id)